In [4]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from sod_model import SODModel
from ipywidgets import FileUpload
from IPython.display import display
import io
from PIL import Image

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
BEST_PATH = 'best_model_unet.pth'
THRESHOLD = 0.5
IMG_SIZE  = 128

model = SODModel().to(DEVICE)
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
model.eval()
print(f"Model loaded on {DEVICE}")
print("Ready! Run the next cell to upload an image.")

Model loaded on cuda
Ready! Run the next cell to upload an image.


In [5]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
from PIL import Image

out = widgets.Output()

def run_inference(change):
    with out:
        clear_output()
        if not uploader.value:
            return
        
        uploaded_file = uploader.value[0]
        img_bytes = uploaded_file['content']
        img_pil = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        img_np = np.array(img_pil)
        
        original = cv2.resize(img_np, (IMG_SIZE, IMG_SIZE))
        tensor = torch.tensor(original / 255.0, dtype=torch.float32)
        tensor = tensor.permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        
        start = time.time()
        with torch.no_grad():
            pred = model(tensor).squeeze().cpu().numpy()
        elapsed = (time.time() - start) * 1000
        
        binary_mask = (pred > THRESHOLD).astype(np.uint8) * 255
        overlay = original.copy()
        mask = pred > THRESHOLD
        if mask.any():
            overlay[mask, 0] = 255
            overlay[mask, 1] = int(overlay[mask, 1].mean() * 0.5)
            overlay[mask, 2] = int(overlay[mask, 2].mean() * 0.5)
        
        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        fig.suptitle(f'Salient Object Detection  |  Inference time: {elapsed:.1f} ms', fontsize=13)
        
        for ax, data, title in zip(
            axes,
            [original, pred, binary_mask, overlay],
            ['Input image', 'Saliency map', 'Binary mask', 'Overlay']
        ):
            ax.imshow(data, cmap='hot' if title == 'Saliency map' else
                            'gray' if title == 'Binary mask' else None)
            ax.set_title(title, fontsize=11)
            ax.axis('off')
        
        plt.tight_layout()
        plt.savefig('demo_output.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Inference time: {elapsed:.1f} ms")
        print(f"Output saved to demo_output.png")

uploader = widgets.FileUpload(accept='image/*', multiple=False)
uploader.observe(run_inference, names='value')
display(uploader, out)
print("Click 'Upload' to select an image from your PC")

FileUpload(value=(), accept='image/*', description='Upload')

Output()

Click 'Upload' to select an image from your PC
